In [ ]:
pip install numpy scipy matplotlib astropy

: 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits

In [ ]:
path = r'D:\Research\NGC247 HI Mapping\VLA-ANGST_HI_NGC247-rob-cube.fits'
ngc247 = fits.open(path)

In [ ]:
header = ngc247[0].header
print(header)
primary = ngc247[0].data
print(primary.shape)

In [ ]:
#isolate the polarisation axis
dummy = primary[0] #1st axis ignore
dummy.shape

In [ ]:
#summing up because we want only velcotiy channel
spectrum = np.sum(dummy, axis=(1,2))
print(spectrum.shape)

In [ ]:
#get paramteric data
CRVAL1 = header['CRVAL1']
CRPIX1 = header['CRPIX1']
CDELT1 = header['CDELT1']
CRVAL2 = header['CRVAL2']
CRPIX2 = header['CRPIX2']
CDELT2 = header['CDELT2']
CRVAL3 = header['CRVAL3']
CRPIX3 = header['CRPIX3']
CDELT3 = header['CDELT3']

In [ ]:
#now we will try to plot the flux for different velocity channels
vel_channels = np.arange(len(spectrum))

#get max values
max_flux = np.max(spectrum)
channel_of_maxflux = np.where(spectrum == max_flux)

plt.figure(figsize=(8,5))
plt.plot(vel_channels, spectrum)
plt.title("Unmasked HI Spectrum of NGC247")
plt.scatter(channel_of_maxflux, max_flux, color='red', label='Max Flux= 23.6 Jy/Beam', s=40)
plt.legend()
plt.axvline(x=channel_of_maxflux, color='pink', linestyle='--')
plt.axhline(y=max_flux, color='pink', linestyle='--')
plt.xlabel("Velocity Channel")
plt.ylabel("Flux (Jy/Beam)")
plt.tight_layout()
plt.show()

In [ ]:
#we can see that there are multiple peaks. So clean it now

noise1 = dummy[0:21,:,:]
noise2 = dummy[116:136,:,:]

noise = np.concatenate([noise1, noise2], axis=0)

rms = np.nanstd(noise)
print(rms)

#sigma fitting
sigma = 3*rms
print(sigma)

In [ ]:
#now letws create a mask and apply to our noisy data
from numpy.ma.core import masked

signal_cube = dummy[21:116,:,:] #thsi is the range where we can see the double-horned profile

#create mask
mask = signal_cube < sigma

#clean the cube with help fo mask
cleaned_cube = np.where(mask,0,signal_cube)

#apply mask on cleaned cube by dropping of pixel axes
masked_spectrum = np.sum(cleaned_cube, axis=(1,2))

In [ ]:
#plot masked spectrum of HI
all_vel = np.arange(len(dummy))
v_full = (CRVAL3+(all_vel-CRPIX3)*CDELT3)/1000

#set all values to zero outisde the range and signal values stays withing masked spectrum
full_spectrum = np.zeros(len(dummy))
full_spectrum[21:116] = masked_spectrum

plt.plot(v_full, full_spectrum)
plt.title("Masked HI Spectrum of NGC247 with 3σ fitting")
plt.xlabel("Flux (Jy/Beam)")
plt.ylabel("Velocity (km/s)")
plt.tight_layout()
plt.show()

In [ ]:
#create moment0 map

moment0 = np.nansum(cleaned_cube*CDELT3, axis=0)


plt.figure(figsize=(12,6))
plt.imshow(moment0, origin='lower', cmap='inferno')
plt.title('Moment0 Map with 3.5σ fit')
plt.xlabel("X-Pixels")
plt.ylabel("Y-Pixels")
plt.colorbar()
plt.tight_layout()
plt.show()

In [ ]:
#now we crewate moment1 map

#first we reshape the v_full which has 95 speeds
v_signal = v_full[21:116]
v_3d = v_signal.reshape(len(v_signal),1,1)
numerator = np.nansum(v_3d * cleaned_cube, axis=0)
denominator = np.nansum(cleaned_cube, axis=0)

#avoid diving by 0 for empty spaces
moment1 = np.divide(numerator, denominator, out=np.full_like(numerator, np.nan), where=(denominator != 0))

plt.figure(figsize=(12,6))
plt.imshow(moment1, origin='lower', cmap='bwr')
plt.title('Moment1 Map with 3σ fit')
plt.xlabel("X-Pixels")
plt.ylabel("Y-Pixels")
plt.colorbar()

In [ ]:
#generate Channel Maps

fig, axes = plt.subplots(5,4, figsize=(14,16), sharex = True, sharey = True)
axes = axes.flatten()


plt.suptitle("Channel Maps of NGC247", fontsize= 20, y=1.0)
#loop the channels
channel_loops = np.linspace(21,115,20, dtype=int)

for i, ch in enumerate(channel_loops):
  ax = axes[i]
  channel_slice = dummy[ch,250:1800,250:1800]
  ax.imshow(channel_slice, vmin=0.05*sigma,cmap='gray_r', origin='lower')
  ax.set_title(f"Channel {ch}")
  ax.set_xlabel("X-Pixels")
  ax.set_ylabel("Y-Pixels")
  current_v = v_full[ch]
  ax.text(0.05, 0.90, f"{current_v:.1f} km/s",
            transform=ax.transAxes, fontsize=10,
            fontweight='bold', bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))
plt.tight_layout()
plt.show()

In [ ]:
BMIN = 1.8173e-3
BMAJ = 1.3295e-3

beam_area_deg = np.pi*BMAJ*BMIN/(4*np.log(2))
beam_area_pixels = beam_area_deg/CDELT2**2
print(beam_area_pixels)

In [ ]:
#clean the moment0
cleaned_cube = np.copy(dummy)

cleaned_cube[cleaned_cube<sigma] = np.nan

moment0_cleaned = np.nansum(cleaned_cube, axis=0)

#calculate now
total_moment0 = np.nansum(moment0_cleaned)
channel_width = 2.58

total_flux = (total_moment0 / beam_area_pixels) * channel_width
distance = 3.5
HI_Sol_Mass = 2.356e05 * (distance**2) * total_flux

print(HI_Sol_Mass)

In [ ]:
print(f" HI Mass of NGC 247: {HI_Sol_Mass:.2e} Solar Masses")

In [ ]:
#moment 2 mapping

#exact number of channels dynamically from signal cleaned cube
num_channels = cleaned_cube.shape[0]

#create the 1D velocity array to match the channel count perfectly
v_1d_corrected = np.arange(num_channels) * 2.578827881

#Broadcast it into 3D so it matches (136, 2048, 2048)
v_3d_corrected = v_1d_corrected[:, np.newaxis, np.newaxis]

#Statistical Moment 2 math with the corrected dimensions
first_term = (v_3d_corrected - moment1) ** 2
second_term = first_term * cleaned_cube
numerator = np.nansum(second_term, axis=0)

#Normalize by Moment 0
moment2 = np.where(moment0_cleaned > 0, numerator / moment0_cleaned, np.nan)


plt.figure(figsize=(10, 8))
plt.imshow(np.sqrt(moment2), cmap='magma', origin='lower')
plt.colorbar(label='Velocity Dispersion $\sigma$ (km/s)')
plt.title('NGC 247: Moment 2 Map (Velocity Dispersion)')
plt.show()

In [ ]:
#calculating moment2
vel_mean = np.mean(v_full)
print(vel_mean)
print(f'Systematic Velocity of NGC 247: {vel_mean:.2f} km/s')

In [ ]:
#using the vel_mean we can now get the redshfited ditance to the galaxy using Hubble's law
c = 3e5  
H0 = 70
V = vel_mean

D = V / H0

print(f'Redshifted Distance to NGC 247: {D:.2f} Mpc')